# Phase 5 — Feature Engineering

Goals:
- convert price from USD to INR
- create required engineered features
- explore additional meaningful features
- document each feature with reason
- save engineered dataset and report

## Imports

In [1]:
from pathlib import Path
import pandas as pd

from src.utils.paths import find_project_root, resolve_project_path
from src.utils.io import save_csv_file, save_text_file
from src.features.build_features import (
    load_feature_input_dataset,
    load_usd_inr_rate,
    add_engineered_features,
    plot_engineered_feature_distributions,
    build_feature_engineering_report,
)

In [2]:
project_root = find_project_root()
project_root

WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation')

## Load cleaned dataset

In [3]:
df = load_feature_input_dataset(project_root)
df.shape, df.head()

((53940, 10),
    carat      cut color clarity  depth  table  price     x     y     z
 0   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
 1   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
 2   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
 3   0.29  Premium     I     VS2   62.4   58.0    334  4.20  4.23  2.63
 4   0.31     Good     J     SI2   63.3   58.0    335  4.34  4.35  2.75)

## Load exchange rate

In [5]:
usd_inr_rate = load_usd_inr_rate(project_root)
usd_inr_rate

93.7908

## Build engineered features

In [6]:
df_fe, feature_doc_df = add_engineered_features(df=df, usd_inr_rate=usd_inr_rate)

print(df_fe.shape)
df_fe.head()

(53940, 22)


,carat,cut,color,clarity,depth,table,price,x,y,z,...,volume_proxy,price_per_carat,dimension_ratio,carat_category,length_width_ratio,depth_pct_from_dimensions,table_depth_interaction,carat_squared,table_to_depth_ratio,face_area
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43,...,38.202030,1417.391304,1.631687,small,0.992462,61.286255,3382.5,0.0529,0.894309,15.7210
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31,...,34.505856,1552.380952,1.673160,small,1.013021,59.767141,3647.8,0.0441,1.020067,14.9376
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31,...,38.076885,1421.739130,1.757576,small,0.995086,56.896552,3698.5,0.0529,1.142355,16.4835
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63,...,46.724580,1151.724138,1.602662,small,0.992908,62.396204,3619.2,0.0841,0.929487,17.7660
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75,...,51.917250,1080.645161,1.580000,small,0.997701,63.291139,3671.4,0.0961,0.916272,18.8790


## Review engineered feature documentation

In [7]:
feature_doc_df

,feature,feature_type,formula,reason,use_for_regression,use_for_clustering,notes
0,price_inr,engineered,price * 93.7908,Converts target price from USD to INR for busi...,False,False,Reporting-only target currency transformation.
1,volume,engineered,x * y * z,Captures a simple geometric size proxy from th...,True,True,Equivalent to volume_proxy in current project ...
2,volume_proxy,engineered,x * y * z,Keeps compatibility with existing project feat...,True,True,Alias of volume.
3,price_per_carat,engineered,price / carat,Useful for descriptive pricing-efficiency anal...,False,False,EDA-only because it is derived from the target.
4,dimension_ratio,engineered,(x + y) / (2 * z),Summarizes average planar spread relative to d...,True,True,Helpful proportion feature.
5,carat_category,engineered,binned carat,Captures non-linear price behavior by size band.,True,True,Categorical engineered feature.
6,length_width_ratio,engineered,x / y,Captures aspect ratio and shape variation.,True,True,Existing candidate feature from project docs.
7,depth_pct_from_dimensions,engineered,(2 * z / (x + y)) * 100,Provides alternate dimensional depth proportion.,True,True,Existing candidate feature from project docs.
8,table_depth_interaction,engineered,table * depth,Captures interaction between two proportion-re...,True,False,More relevant for regression nonlinearity.
9,carat_squared,engineered,carat ** 2,Allows simple nonlinear effect of size.,True,False,Especially useful for linear baselines.


## Inspect selected engineered columns

In [8]:
engineered_cols = [
    "price_inr",
    "volume",
    "volume_proxy",
    "price_per_carat",
    "dimension_ratio",
    "carat_category",
    "length_width_ratio",
    "depth_pct_from_dimensions",
    "table_depth_interaction",
    "carat_squared",
    "table_to_depth_ratio",
    "face_area",
]

df_fe[engineered_cols].head()

,price_inr,volume,volume_proxy,price_per_carat,dimension_ratio,carat_category,length_width_ratio,depth_pct_from_dimensions,table_depth_interaction,carat_squared,table_to_depth_ratio,face_area
0,30575.8008,38.202030,38.202030,1417.391304,1.631687,small,0.992462,61.286255,3382.5,0.0529,0.894309,15.7210
1,30575.8008,34.505856,34.505856,1552.380952,1.673160,small,1.013021,59.767141,3647.8,0.0441,1.020067,14.9376
2,30669.5916,38.076885,38.076885,1421.739130,1.757576,small,0.995086,56.896552,3698.5,0.0529,1.142355,16.4835
3,31326.1272,46.724580,46.724580,1151.724138,1.602662,small,0.992908,62.396204,3619.2,0.0841,0.929487,17.7660
4,31419.9180,51.917250,51.917250,1080.645161,1.580000,small,0.997701,63.291139,3671.4,0.0961,0.916272,18.8790


## Summary stats

In [9]:
df_fe[[
    "volume",
    "price_per_carat",
    "dimension_ratio",
    "length_width_ratio",
    "depth_pct_from_dimensions",
    "carat_squared",
    "table_to_depth_ratio",
    "face_area",
]].describe().T

,count,mean,std,min,25%,50%,75%,max
volume,53940.0,129.910569,78.215151,31.707984,65.204469,114.852720,170.846820,3840.598060
price_per_carat,53940.0,4008.394796,2012.665747,1051.162791,2477.944444,3495.198031,4949.599702,17828.846154
dimension_ratio,53940.0,1.620644,0.050733,0.161478,1.599357,1.617045,1.638199,6.210280
length_width_ratio,53940.0,0.999424,0.011680,0.137351,0.992625,0.995745,1.006944,1.615572
depth_pct_from_dimensions,53940.0,61.755876,2.841358,16.102333,61.042654,61.841181,62.525118,619.279455
carat_squared,53940.0,0.861390,1.056506,0.040000,0.160000,0.490000,1.081600,25.100100
table_to_depth_ratio,53940.0,0.931254,0.048120,0.683625,0.898876,0.923825,0.956240,1.621160
face_area,53940.0,34.119159,13.489469,13.726400,22.278000,32.546400,42.706000,476.501000


## Save engineered feature distribution plot

In [10]:
fig_path = resolve_project_path(
    project_root,
    "figures/feature_engineering/engineered_feature_distributions.png"
)

plot_engineered_feature_distributions(df_fe, fig_path)
print("Saved:", fig_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\feature_engineering\engineered_feature_distributions.png


## Save engineered dataset

In [12]:
feature_engineered_path = resolve_project_path(
    project_root,
    "data/processed/diamonds_feature_engineered.csv"
)

save_csv_file(df_fe, feature_engineered_path, index=False)
print("Saved:", feature_engineered_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\data\processed\diamonds_feature_engineered.csv


## Save interim report draft

In [11]:
report_text = build_feature_engineering_report(
    df=df_fe,
    feature_doc_df=feature_doc_df,
    usd_inr_rate=usd_inr_rate,
    selected_regression_features=[],
    selected_clustering_features=[],
)

report_path = resolve_project_path(
    project_root,
    "reports/feature_engineering_report.md"
)

save_text_file(report_text, report_path)
print("Saved:", report_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\reports\feature_engineering_report.md
